<a href="https://colab.research.google.com/github/nandinichr/My-Recipe_Diary/blob/main/Creative_Game_Finder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

My Recipe Diary

My app helps anyone decide what recipes they can make by taking pictures or entering the groceries or food that already exist in their fridge or pantries.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/MIST 4600/


In [9]:
def creative_type(description):
    description = str(description).lower()

    if "decorat" in description or "renovat" in description or "furniture" in description:
        return "Interior Design"
    elif "architect" in description or "build" in description:
        return "Architecture"
    elif "paint" in description or "draw" in description or "artwork" in description:
        return "Art"
    elif "garden" in description or "landscap" in description:
        return "Landscaping"
    else:
        return "Other"

games["creative_category"] = games["short_description"].apply(creative_type)

creative = games[games["creative_category"] != "Other"].copy()

print("Number of creative games:", len(creative))
print(creative[["name", "creative_category"]].to_string(index=False))

Number of creative games: 140
                                                name creative_category
                                            Terraria      Architecture
                                          The Forest      Architecture
                                           Fall Guys      Architecture
                                             Valheim      Architecture
                                       Hollow Knight               Art
                               Don't Starve Together      Architecture
                                            Palworld      Architecture
                                                Raft      Architecture
                                     Project Zomboid      Architecture
                                          THE FINALS      Architecture
                                  Sons Of The Forest      Architecture
                                    Cities: Skylines      Architecture
                                     Forza Hori

In [10]:
ratings = creative.groupby("creative_category")["pct_pos_total"].mean()

print("Average ratings by creative category:")
print(ratings.sort_values(ascending=False))

top_games = creative.sort_values(
    "pct_pos_total", ascending=False
)

print("\nTop 10 highest-rated creative games:")
print(top_games[
    ["name", "creative_category", "pct_pos_total"]
].head(10))

Average ratings by creative category:
creative_category
Art                93.444444
Interior Design    92.000000
Landscaping        87.857143
Architecture       85.783333
Name: pct_pos_total, dtype: float64

Top 10 highest-rated creative games:
                         name creative_category  pct_pos_total
923      Cats Hidden in Paris               Art             98
154                   Balatro      Architecture             98
6                    Terraria      Architecture             97
36              Hollow Knight               Art             97
98                   Factorio      Architecture             97
957                   Gorogoa               Art             97
241      Dyson Sphere Program      Architecture             97
107            Slay the Spire      Architecture             97
87               Satisfactory      Architecture             97
116  American Truck Simulator       Landscaping             97


In [11]:
paid = creative[
    (creative["price"] > 0) &
    (creative["avg_playtime_hours"] > 0)
].copy()

paid["hours_per_dollar"] = round(
    paid["avg_playtime_hours"] / paid["price"], 2
)

budget = float(input("Enter your maximum budget: $"))

affordable = paid[paid["price"] <= budget]

best_value = affordable.sort_values(
    "hours_per_dollar", ascending=False
)

print("\nTop 10 creative games with the best value:")
print(best_value[
    ["name", "creative_category", "price",
     "avg_playtime_hours", "hours_per_dollar"]
].head(10))

Enter your maximum budget: $20

Top 10 creative games with the best value:
                      name creative_category  price  avg_playtime_hours  \
199                Brotato      Architecture   2.99                39.8   
107         Slay the Spire      Architecture   8.49               110.2   
575           Draw & Guess               Art   2.99                16.1   
175        Space Engineers      Architecture  19.99               104.0   
228          My Summer Car      Architecture  14.99                59.8   
39   Don't Starve Together      Architecture  14.99                54.2   
648   20 Minutes Till Dawn      Architecture   4.99                17.8   
51         Project Zomboid      Architecture  19.99                61.9   
850              Sun Haven      Architecture   9.99                27.7   
213         Gunfire Reborn      Architecture  19.99                52.1   

     hours_per_dollar  
199             13.31  
107             12.98  
575              5.38  
175